# 04 — Pose Keypoint → Exercise Classifier (CV)**Goal:** train a small neural network that takes a flattened MediaPipe pose-keypoint vector (33 keypoints × normalised x/y plus a few derived angles) and classifies it into one of `{squat, push-up, plank, bicep_curl}`. This is what powers `/vision/analyze` and the `/vision/stream` WebSocket — **the model that scores user form in real time**.**Output artifact:** `ai_models/dl_models/exercise_classifier.pth` plus a scaler (`cv_keypoint_scaler.pkl`) and a class-list config (`exercise_classifier_config.json`). Loaded by `backend/cv/exercise_classifier.py`.**Dataset:** `datasets/pose_keypoints.csv` — 9,000+ pre-extracted keypoint rows with the binary/multi-class label in the last column. Each row is a single frame from a webcam capture, MediaPipe-extracted offline.**Why a tiny MLP?** With ≈51-dim input and a small label set, an MLP trains in seconds, runs at >1000 fps on CPU, and avoids the cold-start latency of a CNN. The MediaPipe model upstream is the heavy lifter — this classifier just labels its output.

In [ ]:
import sys, pathlib, json, time, osROOT = pathlib.Path('..').resolve()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport joblibfrom sklearn.preprocessing import StandardScaler, LabelEncoderfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import (accuracy_score, f1_score, classification_report,                             confusion_matrix, ConfusionMatrixDisplay)import torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import DataLoader, TensorDatasetsns.set_style('whitegrid')DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'PyTorch {torch.__version__}  device={DEVICE}')

## 1. Load the dataset

In [ ]:
df = pd.read_csv(ROOT / 'datasets' / 'pose_keypoints.csv')print(f'Loaded {len(df)} rows · {df.shape[1]} columns')print(f'Label distribution:\n{df["label"].value_counts().sort_index()}')# Map label IDs to human names for the configLABEL_NAMES = {0: 'squat', 1: 'pushup', 2: 'plank', 3: 'bicep_curl'}df.head()

## 2. Train/val/test split + standardisation

In [ ]:
X = df.drop(columns=['label']).values.astype(np.float32)y = df['label'].values.astype(np.int64)X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)X_va, X_te, y_va, y_te   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)print(f'train={len(X_tr)}  val={len(X_va)}  test={len(X_te)}')scaler = StandardScaler().fit(X_tr)X_tr_s = scaler.transform(X_tr).astype(np.float32)X_va_s = scaler.transform(X_va).astype(np.float32)X_te_s = scaler.transform(X_te).astype(np.float32)print(f'feature dim: {X_tr_s.shape[1]}')

## 3. The model

In [ ]:
class ExerciseNet(nn.Module):    def __init__(self, in_dim, n_classes, hidden=128, dropout=0.3):        super().__init__()        self.net = nn.Sequential(            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),            nn.Linear(hidden, n_classes),        )    def forward(self, x):        return self.net(x)n_classes = len(np.unique(y))model = ExerciseNet(in_dim=X_tr_s.shape[1], n_classes=n_classes).to(DEVICE)print(model)print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## 4. Training loop

In [ ]:
# DataLoadersbatch_size = 128def loader(X, y, shuffle):    return DataLoader(        TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),        batch_size=batch_size, shuffle=shuffle, num_workers=0,    )train_loader = loader(X_tr_s, y_tr, shuffle=True)val_loader   = loader(X_va_s, y_va, shuffle=False)opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)criterion = nn.CrossEntropyLoss()EPOCHS = 30hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}best_val_acc, best_state = 0.0, Nonefor epoch in range(1, EPOCHS+1):    model.train()    tl = 0.0    for xb, yb in train_loader:        xb, yb = xb.to(DEVICE), yb.to(DEVICE)        opt.zero_grad()        loss = criterion(model(xb), yb)        loss.backward()        opt.step()        tl += loss.item() * xb.size(0)    sched.step()    tl /= len(train_loader.dataset)    model.eval()    vl, correct, total = 0.0, 0, 0    with torch.no_grad():        for xb, yb in val_loader:            xb, yb = xb.to(DEVICE), yb.to(DEVICE)            out = model(xb)            vl += criterion(out, yb).item() * xb.size(0)            correct += (out.argmax(1) == yb).sum().item()            total += yb.size(0)    vl /= total    va = correct / total    hist['train_loss'].append(tl); hist['val_loss'].append(vl); hist['val_acc'].append(va)    if va > best_val_acc:        best_val_acc = va        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:        print(f'[{epoch:02d}/{EPOCHS}] train_loss={tl:.4f} · val_loss={vl:.4f} · val_acc={va:.4f}')model.load_state_dict(best_state)print(f'\nbest val acc: {best_val_acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))axes[0].plot(hist['train_loss'], label='train', color='#00d4ff')axes[0].plot(hist['val_loss'],   label='val',   color='#ff6b35')axes[0].set_title('Cross-entropy loss'); axes[0].legend()axes[0].set_xlabel('epoch')axes[1].plot(hist['val_acc'], color='#00ff88')axes[1].set_title('Validation accuracy'); axes[1].set_xlabel('epoch')plt.tight_layout(); plt.show()

## 5. Test-set evaluation

In [ ]:
model.eval()with torch.no_grad():    logits = model(torch.from_numpy(X_te_s).to(DEVICE)).cpu().numpy()y_pred = logits.argmax(axis=1)acc = accuracy_score(y_te, y_pred)f1  = f1_score(y_te, y_pred, average='macro')print(f'Test accuracy: {acc:.4f}')print(f'Test F1 macro: {f1:.4f}')print()present_labels = sorted(np.unique(np.concatenate([y_te, y_pred])).tolist())target_names   = [LABEL_NAMES.get(int(l), f'class_{l}') for l in present_labels]print(classification_report(y_te, y_pred, labels=present_labels, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_te, y_pred, labels=present_labels)fig, ax = plt.subplots(figsize=(6, 5))ConfusionMatrixDisplay(cm, display_labels=target_names).plot(ax=ax, cmap='magma')plt.title('Test-set confusion'); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 6. Save the artefacts the backend expectsThree files go into `ai_models/dl_models/`:1. `exercise_classifier.pth` — the model weights2. `cv_keypoint_scaler.pkl` — the StandardScaler so inference uses the same normalisation3. `exercise_classifier_config.json` — class list + input dim + arch info

In [ ]:
out_dir = ROOT / 'ai_models' / 'dl_models'out_dir.mkdir(parents=True, exist_ok=True)# 1. weightstorch.save(model.state_dict(), out_dir / 'exercise_classifier.pth')# 2. scalerjoblib.dump(scaler, out_dir / 'cv_keypoint_scaler.pkl')# 3. config — read back by ExerciseClassifier so it can rebuild the architectureclasses = [LABEL_NAMES.get(i, f'class_{i}') for i in range(n_classes)]config = {    'classes':   classes,    'in_dim':    int(X_tr_s.shape[1]),    'hidden':    128,    'dropout':   0.3,    'arch':      'mlp_2x128',    'test_acc':  float(acc),    'test_f1':   float(f1),    'val_acc':   float(best_val_acc),    'trained_at': time.strftime('%Y-%m-%dT%H:%M:%S'),}(out_dir / 'exercise_classifier_config.json').write_text(json.dumps(config, indent=2))print(f'Saved 3 files to {out_dir.relative_to(ROOT)}')

## 7. Smoke-test through the live classifier

In [ ]:
# Reload the way the backend will at startupimport importlib, backend.cv.exercise_classifier as ec_modimportlib.reload(ec_mod)ec_mod.get_classifier.cache_clear() if hasattr(ec_mod.get_classifier, 'cache_clear') else Noneclf = ec_mod.get_classifier()print(f'is_ready={clf.is_ready}  classes={clf.classes}  device={clf.device}')# Use a few real test samples to verify the round-tripsample_idx = np.random.RandomState(0).choice(len(X_te), size=5, replace=False)for i in sample_idx:    raw_kp = X_te[i]  # raw keypoints (BEFORE scaling) — that's what the backend gets from MediaPipe    pred = clf.classify_keypoints(raw_kp.tolist())    truth = LABEL_NAMES.get(int(y_te[i]), str(y_te[i]))    print(f'  truth={truth:12s}  pred={pred.get("exercise","?"):12s}  '          f"conf={pred.get('confidence',0):.3f}")

---The `/vision/analyze` endpoint will now serve form-classified results from this exact model. The MediaPipe pose extractor stays as-is upstream — this notebook only retrains the classifier head that turns keypoints into exercise labels.